In [1]:
! Non-periodic curve - Fortran translation
! Original: K. Moerman 2026 (BASIC dialect)
!
! Plots (f(t2), f(t1)) for t1 = 0..75 step dt, where t2 = offset + t1,
! using f(x) = 250 + 110*(sin(pi*x) + sin(4.5*x)). Because pi and 4.5
! are incommensurate, the curve never exactly repeats ("non-periodic").
! Each frame fades the previous one toward black (alpha ~30/255, like
! the original's translucent background rect) before drawing the new
! curve in white, then "offset" creeps forward by 0.0015.
!
! The original runs forever in a live "while True" loop with REFRESH.
! Since we're writing to files rather than a live screen, that infinite
! loop is replaced by a fixed NFRAMES snapshots written to frames/, one
! PNG per frame - stitch them into a GIF/APNG afterwards to see the
! actual animation (same approach used for the animated Barnsley fern).

program nonperiodic
  implicit none

  integer, parameter :: w = 500, h = 500
  integer, parameter :: nframes = 200          ! how many animation frames to render
  real(kind=8), parameter :: pi = 3.14159265358979323846d0
  real(kind=8), parameter :: dt = 0.03d0
  real(kind=8), parameter :: t1_max = 75.0d0
  real(kind=8), parameter :: alpha = 30.0d0 / 255.0d0   ! colbkg alpha -> fade factor
  real(kind=8), parameter :: fade = 1.0d0 - alpha

  real(kind=8), allocatable :: buf_r(:,:), buf_g(:,:), buf_b(:,:)   ! (x,y), 1-based
  integer, allocatable :: img_r(:,:), img_g(:,:), img_b(:,:)
  real(kind=8) :: offset, t1, t2, px, py
  integer :: frame, k, n_pts, ix, iy
  character(len=256) :: fname

  allocate(buf_r(w, h), buf_g(w, h), buf_b(w, h))
  allocate(img_r(w, h), img_g(w, h), img_b(w, h))
  buf_r = 0.0d0
  buf_g = 0.0d0
  buf_b = 0.0d0

  call execute_command_line('mkdir -p frames')

  offset = 1.0d0
  n_pts = nint(t1_max / dt)

  do frame = 1, nframes
     ! fade previous frame toward black (translucent black "rect" overlay)
     buf_r = buf_r * fade
     buf_g = buf_g * fade
     buf_b = buf_b * fade

     ! draw this frame's curve in white; t2 = offset + t1 since both
     ! start together and step by the same dt each iteration
     do k = 0, n_pts
        t1 = real(k, 8) * dt
        t2 = offset + t1

        px = f(t2)
        py = f(t1)

        ix = nint(px) + 1   ! BASIC's 0-based plot coord -> 1-based Fortran index
        iy = nint(py) + 1

        if (ix >= 1 .and. ix <= w .and. iy >= 1 .and. iy <= h) then
           buf_r(ix, iy) = 255.0d0
           buf_g(ix, iy) = 255.0d0
           buf_b(ix, iy) = 255.0d0
        end if
     end do

     img_r = max(0, min(255, nint(buf_r)))
     img_g = max(0, min(255, nint(buf_g)))
     img_b = max(0, min(255, nint(buf_b)))

     write(fname, '(A,I0.3,A)') 'frames/frame_', frame, '.png'
     call save_png_rgb(trim(fname), img_r, img_g, img_b, w, h)

     offset = offset + 0.0015d0
  end do

  print *, 'Saved ', nframes, ' frames to frames/frame_NNN.png'

contains

  ! non-periodic function: f(x) = 250 + 110*(sin(pi*x) + sin(4.5*x))
  function f(x) result(fx)
    real(kind=8), intent(in) :: x
    real(kind=8) :: fx
    fx = 250.0d0 + 110.0d0 * (sin(pi * x) + sin(4.5d0 * x))
  end function f

  !=========================================================================
  ! Minimal PNG writer (no zlib/libpng dependency), RGB triples supplied
  ! as three separate channel arrays.
  !=========================================================================

  subroutine save_png_rgb(filename, img_r, img_g, img_b, w, h)
    character(len=*), intent(in) :: filename
    integer, intent(in) :: w, h
    integer, intent(in) :: img_r(w, h), img_g(w, h), img_b(w, h)
    integer :: iu
    integer(kind=8) :: crc_table(0:255)
    integer(kind=8) :: raw_size, nfull, rem, nblocks, deflate_size, idat_len
    integer(kind=8) :: crc, adler_a, adler_b, remaining_in_block, total_remaining
    integer :: i, j
    character(len=13) :: ihdr_data

    call build_crc_table(crc_table)

    open(newunit=iu, file=filename, access='stream', form='unformatted', status='replace')

    write(iu) achar(137), achar(80), achar(78), achar(71), &
               achar(13), achar(10), achar(26), achar(10)

    ihdr_data = char_be32(w) // char_be32(h) // achar(8) // achar(2) // &
                achar(0) // achar(0) // achar(0)
    call write_chunk(iu, 'IHDR', ihdr_data, 13, crc_table)

    raw_size = int(h, 8) * int(1 + 3 * w, 8)
    nfull = raw_size / 65535_8
    rem   = raw_size - nfull * 65535_8
    if (rem > 0_8) then
       nblocks = nfull + 1_8
    else
       nblocks = nfull
    end if
    deflate_size = nblocks * 5_8 + raw_size
    idat_len = 2_8 + deflate_size + 4_8

    write(iu) char_be32_8(idat_len)
    write(iu) 'IDAT'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IDAT')

    call emit_byte(iu, 120, crc, crc_table)   ! zlib CMF = 0x78
    call emit_byte(iu, 1,   crc, crc_table)   ! zlib FLG = 0x01

    adler_a = 1_8
    adler_b = 0_8
    remaining_in_block = 0_8
    total_remaining = raw_size

    do j = 1, h
       call emit_raw_byte(iu, 0, crc, crc_table, adler_a, adler_b, &
                           remaining_in_block, total_remaining)   ! filter type: None
       do i = 1, w
          call emit_raw_byte(iu, img_r(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
          call emit_raw_byte(iu, img_g(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
          call emit_raw_byte(iu, img_b(i, j), crc, crc_table, adler_a, adler_b, &
                              remaining_in_block, total_remaining)
       end do
    end do

    call emit_byte(iu, int(iand(ishft(adler_b, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_b, 255_8)),            crc, crc_table)
    call emit_byte(iu, int(iand(ishft(adler_a, -8), 255_8)), crc, crc_table)
    call emit_byte(iu, int(iand(adler_a, 255_8)),            crc, crc_table)

    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    write(iu) char_be32(0)
    write(iu) 'IEND'
    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, 'IEND')
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)

    close(iu)
  end subroutine save_png_rgb

  subroutine emit_byte(iu, byteval, crc, crc_table)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    write(iu) achar(byteval)
    crc = ieor(crc_table(iand(ieor(crc, int(byteval, 8)), 255_8)), ishft(crc, -8))
  end subroutine emit_byte

  subroutine emit_raw_byte(iu, byteval, crc, crc_table, adler_a, adler_b, &
                            remaining_in_block, total_remaining)
    integer, intent(in) :: iu, byteval
    integer(kind=8), intent(inout) :: crc, adler_a, adler_b
    integer(kind=8), intent(inout) :: remaining_in_block, total_remaining
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: block_len, nlen
    logical :: is_last

    if (remaining_in_block == 0_8) then
       block_len = min(65535_8, total_remaining)
       is_last = (total_remaining <= 65535_8)
       call emit_byte(iu, merge(1, 0, is_last), crc, crc_table)
       nlen = 65535_8 - block_len
       call emit_byte(iu, int(iand(block_len, 255_8)),            crc, crc_table)
       call emit_byte(iu, int(iand(ishft(block_len, -8), 255_8)), crc, crc_table)
       call emit_byte(iu, int(iand(nlen, 255_8)),                 crc, crc_table)
       call emit_byte(iu, int(iand(ishft(nlen, -8), 255_8)),      crc, crc_table)
       remaining_in_block = block_len
    end if

    call emit_byte(iu, byteval, crc, crc_table)
    adler_a = mod(adler_a + int(byteval, 8), 65521_8)
    adler_b = mod(adler_b + adler_a, 65521_8)

    remaining_in_block = remaining_in_block - 1_8
    total_remaining = total_remaining - 1_8
  end subroutine emit_raw_byte

  subroutine write_chunk(iu, ctype, data, dlen, crc_table)
    integer, intent(in) :: iu, dlen
    character(len=*), intent(in) :: ctype
    character(len=*), intent(in) :: data
    integer(kind=8), intent(in) :: crc_table(0:255)
    integer(kind=8) :: crc

    write(iu) char_be32(dlen)
    write(iu) ctype
    write(iu) data(1:dlen)

    crc = int(z'FFFFFFFF', 8)
    call crc_update_bytes(crc, crc_table, ctype)
    call crc_update_bytes(crc, crc_table, data(1:dlen))
    crc = ieor(crc, int(z'FFFFFFFF', 8))
    write(iu) char_be32_8(crc)
  end subroutine write_chunk

  subroutine crc_update_bytes(crc, crc_table, s)
    integer(kind=8), intent(inout) :: crc
    integer(kind=8), intent(in) :: crc_table(0:255)
    character(len=*), intent(in) :: s
    integer :: k
    do k = 1, len(s)
      crc = ieor(crc_table(iand(ieor(crc, int(iachar(s(k:k)), 8)), 255_8)), ishft(crc, -8))
    end do
  end subroutine crc_update_bytes

  subroutine build_crc_table(crc_table)
    integer(kind=8), intent(out) :: crc_table(0:255)
    integer(kind=8), parameter :: poly = int(z'EDB88320', 8)
    integer(kind=8) :: c
    integer :: n, k
    do n = 0, 255
       c = int(n, 8)
       do k = 1, 8
          if (iand(c, 1_8) == 1_8) then
             c = ieor(ishft(c, -1), poly)
          else
             c = ishft(c, -1)
          end if
       end do
       crc_table(n) = c
    end do
  end subroutine build_crc_table

  function char_be32(v) result(s)
    integer, intent(in) :: v
    character(len=4) :: s
    integer(kind=8) :: vv
    vv = int(v, 8)
    s = achar(int(iand(ishft(vv, -24), 255_8))) // &
        achar(int(iand(ishft(vv, -16), 255_8))) // &
        achar(int(iand(ishft(vv, -8),  255_8))) // &
        achar(int(iand(vv, 255_8)))
  end function char_be32

  function char_be32_8(v) result(s)
    integer(kind=8), intent(in) :: v
    character(len=4) :: s
    s = achar(int(iand(ishft(v, -24), 255_8))) // &
        achar(int(iand(ishft(v, -16), 255_8))) // &
        achar(int(iand(ishft(v, -8),  255_8))) // &
        achar(int(iand(v, 255_8)))
  end function char_be32_8

end program nonperiodic

 Saved          200  frames to frames/frame_NNN.png
